# Test Stage 5 — Does the droop correction work?

**What this stage checks:** the correction filter. That the measurement window, which sags
towards its edges because averaging is not even-handed, comes out flat once the correction is
applied.

This is a **before-and-after** measurement, and that is what makes it convincing: the same
signal, the same hardware, the same sweep, run twice with one register bit changed.

---

## The problem being fixed

Averaging treats frequencies unequally. A signal at the centre of the window passes at full
strength; one near the edge comes out weaker. The same ion signal would therefore appear
smaller simply for sitting further from the reference tone — a systematic error that depends
on where the ion happens to be.

Predicted, from the filter design:

| Position in the window | Uncorrected | Corrected |
|---|---|---|
| centre | 0 dB | 0 dB |
| halfway out | −0.573 dB | −0.001 dB |
| band edge (0.20 × output rate) | **−2.317 dB** | **−0.043 dB** |

So the correction should reduce the variation across the band from about 2.3 dB to under
0.05 dB — better than fiftyfold.

## How the sweep works, without touching the generator

Rather than retuning the signal generator at every point, **we move the reference tone.**
Shifting the reference by Δ moves the signal to a position −Δ within the window. The generator
stays where it is, and the sweep is entirely electronic, which is both faster and free of the
generator's own settling and amplitude drift.

## What you need

| | |
|---|---|
| Stages 1–4 | Passing |
| Signal generator | Sine into **IN1**, a few hundred mV, frequency noted below |
| Decimation | **R ≥ 256.** The filter reuses one multiplier 155 times per sample, so it needs about 160 clock cycles between samples. Below R = 256 it is bypassed automatically and this stage cannot run |

## Setup

In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK = 125e6          # FPGA processing clock [Hz]

# =============================================================================
#  Acquisition system driver  -  register block at 0x40600000 (sys[6])
# =============================================================================
#  CRITICAL RULE: every access is a SINGLE 32-bit read or write. A bulk mmap
#  slice becomes a burst transfer the bus bridge cannot acknowledge; it locks
#  the bus and reboots the board. Do not "optimise" these loops into slices.
#
#  NOTE ON NUMPY: the board's numpy is old. Use np.polyfit rather than
#  np.linalg.lstsq(..., rcond=None), which fails with "a float is required".
# =============================================================================
ACQ_BASE = 0x40600000

R_CTRL       = 0x00   # [0]en [1]soft-rst [2]test [3]tap [4]arm [5]wrap [6]fir-byp [7]dma-clr
R_LO_FINC    = 0x04
R_DEC_LOG2   = 0x08
R_SRC_SEL    = 0x0C
R_TAP_SEL    = 0x10   # 0=mixer 1=decimator 2=corrector
R_TAP_SHIFT  = 0x14
R_PEEK_I     = 0x18
R_PEEK_Q     = 0x1C
R_SAMPLE_CNT = 0x20
R_STATUS     = 0x24
R_MEM_START  = 0x28
R_MEM_STOP   = 0x2C
R_MEM_CUR    = 0x30
R_DMA_COUNT  = 0x34
R_STATUS2    = 0x38   # [0]mem overflow [1]filter overrun [2]filter bypassed
R_COEF_ADDR  = 0x40   # which correction-filter coefficient to look at, 0..154
R_COEF_DATA  = 0x44   # write: store it.  read: what the chip actually holds

NHALF     = 155       # coefficients stored (the unique half)
NTAPS     = 309       # taps in the whole filter
COEF_FRAC = 17        # coefficients are integers scaled by 2**17
NSTAGE    = 4         # decimator stages, which sets the droop


def s32(v):
    """Reinterpret a 32-bit register value as signed."""
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val): self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):      return struct.unpack('<I', self.m[off:off+4])[0]

    @staticmethod
    def finc(f_lo):
        """f_out = f_clk * finc / 2**32, so finc = f_out/f_clk * 2**32."""
        return int(round(f_lo / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    def setup(self, f_lo, dec_log2, src=0, test=False, tap_en=False,
              tap_sel=0, tap_shift=10, fir_bypass=False,
              arm=False, wrap=True, mem_start=None, mem_stop=None):
        """Configure and start. ORDER MATTERS: the decimator holds a running
        total that assumes one fixed decimation factor, so always
        disable -> write settings -> reset -> enable."""
        self.w(R_CTRL, 0)                                  # 1. disable
        self.w(R_LO_FINC,   self.finc(f_lo))               # 2. settings
        self.w(R_DEC_LOG2,  dec_log2)
        self.w(R_SRC_SEL,   src)
        self.w(R_TAP_SEL,   tap_sel)
        self.w(R_TAP_SHIFT, tap_shift)
        if mem_start is not None: self.w(R_MEM_START, mem_start)
        if mem_stop  is not None: self.w(R_MEM_STOP,  mem_stop)
        ctrl = (1 |
                (4  if test       else 0) |
                (8  if tap_en     else 0) |
                (16 if arm        else 0) |
                (32 if wrap       else 0) |
                (64 if fir_bypass else 0))
        self.w(R_CTRL, ctrl | 2 | 128)                     # 3. enable + reset + dma clear
        self.w(R_CTRL, ctrl)                               # 4. running

    def disable(self):  self.w(R_CTRL, 0)
    def peek(self):     return s32(self.r(R_PEEK_I)), s32(self.r(R_PEEK_Q))
    def count(self):    return self.r(R_SAMPLE_CNT)

    def read_coeffs(self, n=NHALF):
        """Read the correction filter out of the chip, one coefficient at a time.

        This reads what the chip ACTUALLY HOLDS, which is not the same thing as
        what it was supposed to hold. Telling those two apart is the whole point.

        The hardware sign-extends each 18-bit coefficient to 32 bits before
        handing it over, so undo a 32-BIT sign here (s32), not an 18-bit one.
        Undoing the wrong width makes every negative coefficient come back as a
        huge positive number - about 2**18 too big.
        """
        out = []
        for a in range(n):
            self.w(R_COEF_ADDR, a)
            out.append(s32(self.r(R_COEF_DATA)))
        return np.array(out, dtype=float)

    def status(self):
        s = self.r(R_STATUS2)
        return dict(mem_overflow=bool(s & 1),
                    filter_overrun=bool(s & 2),
                    filter_bypassed=bool(s & 4))


acq = Acq()
print('acquisition system mapped at 0x%08X' % ACQ_BASE)
st = acq.status()
print('status: memory overflow=%s  filter overrun=%s  filter bypassed=%s'
      % (st['mem_overflow'], st['filter_overrun'], st['filter_bypassed']))

In [ ]:
# ===========================================================================
#  MEASUREMENT
# ===========================================================================
#  Two rules here, both learned the hard way on 2026-07-29.
#
#  1. AMPLITUDE IS MEASURED FROM ONE REGISTER, NOT TWO.
#     |I + jQ| needs two reads microseconds apart, and a new sample arrives
#     every 8 microseconds, so they often come from different samples. The
#     result is biased low by up to 0.9 dB and jitters by up to 48%, both
#     growing with offset frequency - the same shape as the droop being looked
#     for. The test was measuring itself.
#
#  2. THE READS ARE THEMSELVES A SAMPLING PROCESS, AND THEY ARE SLOW.
#     Python manages roughly a thousand reads a second, so anything above about
#     500 Hz folds back and reads as something slower. A 1366 Hz beat came back
#     as 446 Hz. Any method that counts cycles is therefore limited to a few
#     hundred hertz, and the old two-point beat trick - which deliberately moved
#     the reference by 1220 Hz - was measuring an alias and calling it a fault.
#
#  So the generator's offset is found from the PHASE of I + jQ rather than from
#  counting anything. Phase is immune to both problems, and gives the sign for
#  free. See offset_hz below.
# ===========================================================================
COEFW = 18
RAIL  = (1 << 31) - 1
NOISE_FLOOR = 2.0e5          # generous; the real floor is a few times 1e4


def read_I(n=1500, gap=0.001):
    """Read the I sample alone, repeatedly. Returns the values and the span."""
    v, t0 = [], time.time()
    for _ in range(n):
        v.append(s32(acq.r(R_PEEK_I)))
        if gap:
            time.sleep(gap)
    return np.array(v, dtype=float), time.time() - t0


def read_IQ(n=3000, gap=0.0002):
    """Read I and Q together, with a timestamp for each pair.

    The two reads are microseconds apart, which ruins their MAGNITUDE - but it
    adds the same small constant to every phase, and a constant cancels out of
    the phase DIFFERENCE that offset_hz measures. So this is unsuitable for
    amplitude and perfectly suited to frequency.
    """
    t, i, q = [], [], []
    for _ in range(n):
        t.append(time.time())
        i.append(s32(acq.r(R_PEEK_I)))
        q.append(s32(acq.r(R_PEEK_Q)))
        if gap:
            time.sleep(gap)
    return (np.array(t) - t[0], np.array(i, dtype=float), np.array(q, dtype=float))


def amplitude(v):
    """Size of the signal, whether it is standing still or going round.

        amplitude = sqrt( mean^2 + 2 * variance )

    A constant has no variance, so this gives its value. A sine has no mean, so
    this gives sqrt(2) x its spread, which is its amplitude. No special case at
    zero offset, which matters because zero offset is never exactly zero.
    """
    return float(np.sqrt(v.mean() ** 2 + 2 * v.var()))


def measure(f_lo, dec_log2, bypass, settle=0.3, n=1500, gap=0.001):
    """Returns (amplitude, mean, spread, min, max)."""
    acq.setup(f_lo=f_lo, dec_log2=dec_log2, fir_bypass=bypass)
    time.sleep(settle)
    v, span = read_I(n, gap)
    return amplitude(v), v.mean(), v.std(), v.min(), v.max()


def report(label, f_lo, dec_log2, bypass=False):
    """One measurement, printed as a line. Returns the amplitude."""
    a, m, s, lo, hi = measure(f_lo, dec_log2, bypass)
    print('  %-22s amplitude %13.1f   mean %13.1f   min %13.1f   max %13.1f'
          % (label, a, m, lo, hi))
    return a


def offset_hz(f_lo, dec_log2, bypass=True, settle=0.3, n=3000, gap=0.0002):
    """How far the signal is from the reference tone, SIGN INCLUDED.

    The downconverted signal is a vector going round at the difference
    frequency. Its angle therefore advances by a constant amount per second,
    and that rate IS the difference - positive one way round, negative the
    other. Fit a straight line to the angle against time and read off the slope.

    Why this beats counting cycles:
      * it gives the sign, which counting never can
      * it does not care that I and Q are read microseconds apart, because a
        fixed lag adds the same constant to every angle and cancels in the slope
      * how well the straight line fits tells you whether this is a tone at all.
        A real tone gives a residual of a few hundredths of a radian. Noise has
        no coherent phase and gives something near a radian.

    Returns (offset in Hz, residual in radians, samples per second achieved).
    The one limit is that the angle must not advance more than half a turn
    between reads, which caps it at about half the read rate.
    """
    acq.setup(f_lo=f_lo, dec_log2=dec_log2, fir_bypass=bypass)
    time.sleep(settle)
    t, i, q = read_IQ(n, gap)
    rate = (len(t) - 1) / max(t[-1], 1e-9)

    ph = np.unwrap(np.angle(i + 1j * q))
    slope, icept = np.polyfit(t, ph, 1)          # radians per second
    resid = float(np.std(ph - (slope * t + icept)))
    return slope / (2 * np.pi), resid, rate


# ---------------------------------------------------------------------------
#  IS THERE A SIGNAL, AND WHERE EXACTLY IS IT?
#
#  Two failures look identical from inside a test: nothing plugged in, and a
#  frequency typed in wrongly. The window is only +-24 kHz out of a 62 MHz
#  input range, so being 9 MHz out removes the signal as completely as pulling
#  the cable. Both must be caught, and told apart.
# ---------------------------------------------------------------------------
def require_signal(f_nom, dec_log2, search=True):
    """Refuse to continue without a signal. Returns its true frequency."""
    a, _, _, _, _ = measure(f_nom, dec_log2, True, n=800)
    print('  amplitude                     : %14.1f counts' % a)

    if a < NOISE_FLOOR:
        print('')
        print('*** NO SIGNAL ***')
        print('')
        print('  %.0f counts is the noise floor. A connected generator gives millions.' % a)
        print('  Check, in this order:')
        print('    1. the generator OUTPUT is switched on - not just the generator')
        print('    2. the cable is in IN1, not IN2 or OUT1')
        print('    3. the level is a few hundred mV, jumper on LV')
        print('    4. swap the cable')
        if search:
            print('')
            print('  Sweeping the whole input range before giving up:')
            find_generator()
        raise SystemExit('stopping: nothing on the input')

    d, resid, rate = offset_hz(f_nom, dec_log2)
    print('  reads per second              : %14.0f   (this caps what can be seen)' % rate)
    print('  phase fit residual            : %14.3f rad (a tone gives << 1)' % resid)
    print('  offset from what you typed    : %+14.1f Hz  (%+.2f ppm)'
          % (d, 1e6 * d / f_nom))

    if resid > 0.5:
        print('')
        print('*** THERE IS SOMETHING THERE, BUT IT IS NOT A STEADY TONE ***')
        print('')
        print('  The signal is strong enough, but its phase does not advance at a')
        print('  constant rate, so it is not a single frequency. Either the offset')
        print('  is faster than %.0f Hz - more than half the read rate, which folds' % (rate / 2))
        print('  and cannot be untangled - or what is in the window is noise.')
        if search:
            print('')
            print('  Sweeping the whole input range:')
            find_generator()
        raise SystemExit('stopping: no steady tone in the window')

    #  Confirm it, by tuning to where we think it is and checking what is left.
    f_true = f_nom + d
    d2, resid2, _ = offset_hz(f_true, dec_log2)
    print('  after correcting, offset is   : %+14.1f Hz  (should be near zero)' % d2)
    if abs(d2) > max(20.0, 0.1 * abs(d)):
        print('  ...which it is not. The generator is drifting while we measure.')
    print('')
    print('  Signal present at %.4f MHz.' % (f_true / 1e6))
    return f_true


def find_generator(fmin=0.2e6, fmax=60.0e6, quiet=False):
    """Sweep the reference tone across the input range and report what is there.

    Coarse to fine: a small decimation gives a wide window, so few steps cover
    everything; then narrow down. About twenty seconds for the whole range.
    """
    if not quiet:
        print('    searching %.1f - %.1f MHz ...' % (fmin / 1e6, fmax / 1e6))

    f_o = FS_CLK / (1 << 6)
    best_a, best_f = 0.0, None
    f = fmin
    while f < fmax:
        a, _, _, _, _ = measure(f, 6, True, settle=0.02, n=250, gap=0)
        if a > best_a:
            best_a, best_f = a, f
        f += 0.25 * f_o

    if best_f is None or best_a < NOISE_FLOOR:
        if not quiet:
            print('    nothing anywhere in that range (largest %.0f counts).' % best_a)
        return None
    if not quiet:
        print('    something near %.3f MHz (%.0f counts)' % (best_f / 1e6, best_a))

    for dec, span, nstep in ((8, 0.8e6, 17), (10, 0.10e6, 21)):
        vals, cand = [], np.linspace(best_f - span, best_f + span, nstep)
        for f in cand:
            a, _, _, _, _ = measure(f, dec, True, settle=0.05, n=400, gap=0)
            vals.append(a)
        best_f = cand[int(np.argmax(vals))]
        if not quiet:
            print('    refined at R = %-5d : %.4f MHz (%.0f counts)'
                  % (1 << dec, best_f / 1e6, max(vals)))

    d, resid, _ = offset_hz(best_f, 10)
    if resid < 0.5:
        best_f = best_f + d

    if not quiet:
        print('')
        print('    >>> The generator is at %.4f MHz.' % (best_f / 1e6))
        print('        Put this in the setup cell above and run again:')
        print('')
        print('            F_SIG = %.0f' % best_f)
        print('')
    return best_f


---
## Step 1 — Confirm the filter is actually in circuit

The correction is skipped automatically when the decimation factor is too small. Before
measuring anything, check the hardware agrees it is running.

**Passes if:** `filter bypassed` reads False with R = 1024, and True with R = 64.

That second half matters as much as the first: it confirms the automatic bypass works, so the
filter can never be silently fed samples faster than it can process them.

In [ ]:
# ===========================================================================
#  THE ONE LINE YOU MUST EDIT
#
#  F_SIG is what your signal generator is set to. The measurement window is only
#  +-24 kHz wide out of a 62 MHz input range, so a wrong value here does not
#  make the signal weak - it makes it VANISH, exactly as if the cable were
#  unplugged. Getting this wrong cost two full runs on 2026-07-29.
#
#  If it is wrong, the test now says so and sweeps the whole range to tell you
#  the right value. It does not guess silently.
# ===========================================================================
F_SIG = 10.0e6         # <-- YOUR SIGNAL GENERATOR FREQUENCY [Hz]
DEC   = 10             # log2 of the decimation factor. 10 -> R = 1024
R     = 1 << DEC

f_out = FS_CLK / R
edge  = 0.20 * f_out                 # the band the correction is designed for

print('decimation R          : %d' % R)
print('output sample rate    : %.2f kHz' % (f_out / 1e3))
print('corrected band edge   : +-%.2f kHz' % (edge / 1e3))
print()

for L, note in ((DEC, 'the setting this stage uses'), (6, 'R = 64, too fast for the filter')):
    acq.setup(f_lo=F_SIG, dec_log2=L)
    time.sleep(0.1)
    st = acq.status()
    print('R = %5d : filter bypassed = %-5s   (%s)' % (1 << L, st['filter_bypassed'], note))

acq.setup(f_lo=F_SIG, dec_log2=DEC)
time.sleep(0.1)
ok = (not acq.status()['filter_bypassed'])
print()
print('STEP 1:', 'PASS' if ok else 'FAIL - the filter is not in circuit at R = %d' % R)
print()

# Check there is something to measure BEFORE spending two minutes measuring it.
print('Is there a signal on the input?')
F_TRUE = require_signal(F_SIG, DEC)

---
## Step 1b — Read the filter back out of the chip

Everything else in this stage measures the filter *indirectly*, through a signal, and so
depends on the measurement being right. This step does not. It reads all 155 coefficients out
of the chip's memory and works out, on paper, what filter they make.

**It separates two things that look identical from the outside:**

| | |
|---|---|
| The chip holds the wrong numbers | Nothing downstream can be trusted. Fix this first |
| The chip holds the right numbers | The filter is fine, and any failure below is in how the signal is measured |

Two ways the numbers can be wrong. Either they never got in (the generated file was not picked
up during the build), or **something overwrote them at run time** — the FIR Filter Designer
notebook writes to this same memory, and what it writes stays there until the board is power
cycled or the bitstream reloaded. If you have used the designer since the last reload, reload
the bitstream before trusting anything below.

**Passes if:** every coefficient matches `fir_coeffs.txt`, and the response computed from them
flattens the droop to under 0.05 dB.


In [ ]:
# ===========================================================================
#  Read the filter out of the chip and check it on paper.
# ===========================================================================
import os

chip = acq.read_coeffs(NHALF)                       # the unique half, 0..154
full = np.concatenate([chip, chip[-2::-1]])         # mirror it: 155 + 154 = 309

print('read %d coefficients from the chip' % NHALF)
print('  first three : %s' % full[:3])
print('  centre tap  : %.0f' % full[NTAPS // 2])
print('  sum         : %.0f      (the design forces exactly %d)'
      % (full.sum(), 1 << COEF_FRAC))
print('  DC gain     : %.6f   (must be 1.000000: turning the correction on'
      % (full.sum() / (1 << COEF_FRAC)))
print('                             must not change the scale of your data)')
print()

# ---- compare against the file the build was made from, if it is here --------
ref_path = 'fir_coeffs.txt'
have_ref = os.path.exists(ref_path)
if have_ref:
    ref = np.loadtxt(ref_path).astype(np.int64)
    if len(ref) != NTAPS:
        print('%s has %d values, expected %d - skipping the comparison'
              % (ref_path, len(ref), NTAPS))
        match = None
    else:
        bad = np.nonzero(full != ref)[0]
        match = (len(bad) == 0)
        if match:
            print('MATCH: all %d taps are exactly what gen_fir_coeffs.py produced.' % NTAPS)
        else:
            print('MISMATCH: %d of %d taps differ from %s' % (len(bad), NTAPS, ref_path))
            for k in bad[:8]:
                print('    tap %3d : chip %12.0f    file %12.0f' % (k, full[k], ref[k]))
            if len(bad) > 8:
                print('    ... and %d more' % (len(bad) - 8))
            print()
            print('    Either the build did not pick up fir_coeffs.vh, or something has')
            print('    written over the memory since. The FIR Filter Designer notebook')
            print('    writes here; its changes survive until the bitstream is reloaded.')
            print('    Reload the bitstream and run this cell again before going further.')
else:
    print('(%s not found next to this notebook - skipping the file comparison)' % ref_path)
    match = None
print()

# ---- what response do these numbers actually give? -------------------------
#  Evaluated at the frequencies we are about to sweep, in units of the output
#  sample rate. The decimator's droop is sinc(u)**NSTAGE; a perfect corrector
#  is its reciprocal, so the product should be flat.
u   = np.linspace(-0.20, 0.20, 401)
n   = np.arange(NTAPS)
H   = np.abs(full.dot(np.exp(-2j * np.pi * np.outer(n, u)))) / (1 << COEF_FRAC)
D   = np.abs(np.sinc(u)) ** NSTAGE
cdb = 20 * np.log10(H * D)
udb = 20 * np.log10(D)

print('predicted from the coefficients now in the chip, across |f| <= 0.20 x f_out:')
print('  droop, uncorrected : %6.3f dB' % (udb.max() - udb.min()))
print('  after correction   : %6.3f dB' % (cdb.max() - cdb.min()))
print()

paper_ok = (cdb.max() - cdb.min()) < 0.05 and abs(full.sum() / (1 << COEF_FRAC) - 1) < 1e-6
print('STEP 1b:', 'PASS' if paper_ok else 'FAIL')
if paper_ok:
    print('   The chip holds the intended filter. It is arithmetically incapable of')
    print('   producing more than 0.05 dB of ripple, so if Step 2 below reports')
    print('   several decibels, the fault is in the MEASUREMENT, not the filter.')
else:
    print('   The chip does not hold the intended filter. Nothing below will pass')
    print('   until this does. Reload the bitstream and try again.')


In [ ]:
# ===========================================================================
#  DIAGNOSTIC - is the zero-offset reading trustworthy?
#
#  Step 1b already settled whether the chip holds the right filter. This cell
#  settles the other half: whether the MEASUREMENT at zero offset is sound.
#
#  Read the `spread` column. Everywhere else in the sweep the signal spins and
#  its magnitude is rock steady, so the spread is a fraction of a percent. At
#  exactly zero offset the signal stops spinning and lands on top of the
#  mixer's own steady offset; the reading then drifts, and the spread gives it
#  away.
#
#  If the spread at u = 0 is large compared with the others, that single
#  reading is unusable - and the old version of Step 2 divided every other
#  point by it, which is what turned a fraction of a decibel into thirty.
# ===========================================================================
print('%9s %6s %14s %10s %14s %10s %10s'
      % ('u=f/f_out', 'corr', 'magnitude', 'spread', '', '', 'on/off'))
print('-' * 78)

rows = []
for u in (0.000, 0.002, 0.005, 0.010, 0.020, 0.040, 0.060, 0.120):
    r = {}
    for byp in (True, False):
        a, _, sd, _, _ = measure(F_TRUE - u * f_out, DEC, byp, n=600)
        r['off' if byp else 'on'] = (a, sd)
    rows.append((u, r))
    print('%9.3f %6s %14.1f %9.1f (%4.1f%%)'
          % (u, 'OFF', r['off'][0], r['off'][1], 100 * r['off'][1] / max(r['off'][0], 1e-9)))
    print('%9s %6s %14.1f %9.1f (%4.1f%%) %22.3f'
          % ('', 'ON', r['on'][0], r['on'][1], 100 * r['on'][1] / max(r['on'][0], 1e-9),
             r['on'][0] / max(r['off'][0], 1e-9)))

print()
spread0 = max(rows[0][1]['on'][1] / max(rows[0][1]['on'][0], 1e-9),
              rows[0][1]['off'][1] / max(rows[0][1]['off'][0], 1e-9))
spread_elsewhere = np.mean([max(r['on'][1] / max(r['on'][0], 1e-9),
                                r['off'][1] / max(r['off'][0], 1e-9))
                            for _, r in rows[1:]])
print('relative spread at zero offset : %.2f %%' % (100 * spread0))
print('relative spread elsewhere      : %.2f %%' % (100 * spread_elsewhere))
print()
if spread0 > 5 * max(spread_elsewhere, 1e-4):
    print('>>> The zero-offset reading is unstable, as expected. It is contaminated')
    print('    by the mixer offset and must not be used as a reference. Step 2 below')
    print('    does not use it.')
else:
    print('>>> The zero-offset reading looks stable, so the mixer offset is small')
    print('    here. Step 2 will still avoid it, which costs nothing.')
print()
print('The design predicts on/off rising smoothly from 1.000 at u = 0 to about')
print('1.30 at u = 0.20. Anything else, with Step 1b passing, is a measurement fault.')


---
## Step 2 — Sweep the window, correction off then on

Step the reference tone across the window and record the signal's amplitude at each position.
Twice: once with the correction bypassed, once with it active. Nothing changes between the two
runs but one register bit.

### Read the `on/off` column first

It is the ratio of the two runs **at the same frequency**, so everything common to them cancels:
the droop, the generator's amplitude, the cable, the converter, and any bias in how the
amplitude is worked out. What is left is the correction filter's own response, and it can be
compared straight against the filter that is loaded — the `designed` column beside it.

If those two columns agree, the filter is doing exactly what it was built to do, whatever the
two curves either side of them look like. On 2026-07-29 they agreed to an rms of **0.027** while
the two individual curves were jagged by more than a decibel. The jaggedness was the
measurement.

### Three things this sweep deliberately does

**It reads one register, not two.** `|I + jQ|` needs two reads microseconds apart; at higher
offsets they land on different samples and the result is biased low by up to 0.9 dB *and* jitters
by up to 48%, both growing with offset — the same shape as the droop being looked for.

**It never measures at exactly zero offset.** There the wanted signal stops moving and becomes
indistinguishable from the mixer's own steady offset.

**It finds the generator first.** The generator is not locked to the board, and the size of the
gap between them is measurable from one channel but its **sign** is not. `signed_offset()` gets
the sign by measuring twice.

**Passes if:** the uncorrected curve varies by roughly the predicted droop, and the corrected one
is flat to a few tenths of a decibel — with the `on/off` column matching the designed response,
which is the strict test.


In [ ]:
# Sweep positions, as a fraction of the output sample rate. 0.20 is the edge of
# the band the correction was designed for; we go slightly beyond to show where
# it stops working. The gap around zero is deliberate - see the markdown above.
GAP  = 0.015
half = np.linspace(GAP, 0.24, 12)
frac = np.concatenate([-half[::-1], half])

# F_TRUE was established in Step 1 by require_signal(), which also proved there
# IS a signal. Re-measured here only if this cell is run on its own.
try:
    F_TRUE
except NameError:
    print('Locating the generator:')
    F_TRUE = require_signal(F_SIG, DEC)
    print()
offsets = frac * f_out


def sweep(bypass):
    """Amplitude versus position in the window. Moving the reference tone by
    -offset places the signal at +offset within the window."""
    amps, spreads = [], []
    for off in offsets:
        a, m, s, lo, hi = measure(F_TRUE - off, DEC, bypass)
        amps.append(a); spreads.append(s / max(a, 1e-9))
    return np.array(amps), np.array(spreads)


print('sweeping with the correction OFF ...')
off_curve, off_spread = sweep(bypass=True)
print('sweeping with the correction ON  ...')
on_curve,  on_spread  = sweep(bypass=False)

if acq.status()['filter_overrun']:
    print()
    print('WARNING: the filter reported an overrun. R is too small for it; raise DEC.')


def to_db(curve):
    """dB, with the zero line taken from a smooth fit rather than one reading.

    The response is even in frequency and gentle, so a straight line in frac**2
    describes the middle of it well. Extrapolating that fit to frac = 0 gives a
    reference built from every point, which no single bad reading can move.
    """
    d = 20 * np.log10(curve)
    inner = np.abs(frac) <= 0.12
    a, b = np.polyfit(frac[inner] ** 2, d[inner], 1)      # d ~ a*frac**2 + b
    return d - b


off_db = to_db(off_curve)
on_db  = to_db(on_curve)
ratio  = on_curve / off_curve          # the filter's own response, point by point

print()
print('%10s %11s %11s %12s %11s' % ('f / f_out', 'off [dB]', 'on [dB]', 'on/off', 'designed'))
print('-' * 60)
n_t  = np.arange(NTAPS)
full = np.concatenate([ref[:NHALF], ref[:NHALF][-2::-1]]).astype(float) if have_ref else None
for k in range(0, len(frac), 2):
    want = (abs(full.dot(np.exp(-2j * np.pi * abs(frac[k]) * n_t))) / (1 << COEF_FRAC)
            if full is not None else float('nan'))
    print('%10.3f %11.3f %11.3f %12.3f %11.3f'
          % (frac[k], off_db[k], on_db[k], ratio[k], want))

band = np.abs(frac) <= 0.20
var_off = off_db[band].max() - off_db[band].min()
var_on  = on_db[band].max()  - on_db[band].min()
pred    = 20 * np.log10(np.abs(np.sinc(frac[band])) ** NSTAGE)
pred_off = pred.max() - pred.min()

print()
print('variation across the corrected band (|f| <= 0.20 x f_out):')
print('  correction OFF : %6.3f dB   (predicted %.3f at these points)'
      % (var_off, pred_off))
print('  correction ON  : %6.3f dB   (predicted under 0.05)' % var_on)
print('  improvement    : %6.1fx' % (var_off / max(var_on, 1e-6)))
print()

# How well does the measured filter response match the designed one? This is a
# ratio taken at ONE frequency, so anything common to both curves cancels - it
# is the strictest test here and the least susceptible to measurement error.
if full is not None:
    want = np.array([abs(full.dot(np.exp(-2j * np.pi * abs(x) * n_t))) / (1 << COEF_FRAC)
                     for x in frac])
    rms = float(np.sqrt(np.mean((ratio - want) ** 2)))
    print('measured filter response vs designed : rms %.4f over %d points'
          % (rms, len(frac)))
    print()

# ---------------------------------------------------------------------------
#  WHICH NUMBER DECIDES THIS
#
#  Not the flatness of the corrected curve. That is a max minus a min over 21
#  noisy points, and the range of 21 samples of anything is about 3.7 times its
#  own scatter - so with roughly 1% per point it comes out near 0.3 dB however
#  perfect the filter is. Judging the filter by it means judging the sample
#  count.
#
#  The on/off ratio is taken at ONE frequency, so the droop, the generator, the
#  cable and any bias in the amplitude estimate all cancel. Comparing it against
#  the filter actually loaded is the strict test, and it is what decides.
#
#  Expected scatter on one such ratio, from the amplitude estimator: about 1.3%
#  at these offsets (see the note in Step 3). Three times that is the tolerance.
# ---------------------------------------------------------------------------
RMS_TOL = 0.05
ok = (var_off > 0.7 * pred_off) and (full is None or rms < RMS_TOL)
print('STEP 2:', 'PASS' if ok else 'FAIL')
if full is not None:
    print('   filter response vs designed : rms %.4f   (tolerance %.4f)' % (rms, RMS_TOL))
    print('   window droop, uncorrected   : %.3f dB (predicted %.3f)' % (var_off, pred_off))
    print('   corrected curve flatness    : %.3f dB - informational only, see above'
          % var_on)
if var_off < 0.7 * pred_off:
    print('   the uncorrected curve is too flat - is the bypass bit reaching the')
    print('   hardware? Check filter_bypassed in the status register changes.')
if full is not None and rms >= RMS_TOL:
    print('   The measured filter response does not match the filter that is loaded.')
    print('   That is a real disagreement, not scatter. Run Test Stage 5b, which')
    print('   tests the same hardware with filters whose answer is known.')


In [ ]:
import matplotlib.pyplot as plt

u   = np.linspace(-0.24, 0.24, 401)
ideal_droop = 20 * np.log10(np.abs(np.sinc(u)) ** NSTAGE)

plt.figure(figsize=(9, 5))
plt.plot(u, ideal_droop, 'k--', lw=1, label='droop, predicted')
plt.plot(frac, off_db, 'o-', label='correction OFF, measured')
plt.plot(frac, on_db,  's-', label='correction ON, measured')
plt.axvspan(-0.20, 0.20, color='0.92', zorder=0, label='band the filter corrects')
plt.axhline(0, color='0.6', lw=0.8)
plt.xlabel('position in the window  [fraction of the output sample rate]')
plt.ylabel('amplitude relative to the centre [dB]')
plt.title('Droop before and after correction')
plt.legend(); plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()

plt.figure(figsize=(9, 4))
plt.plot(frac, on_db, 's-')
plt.axvspan(-0.20, 0.20, color='0.92', zorder=0)
plt.axhline(0, color='0.6', lw=0.8)
plt.ylim(-0.5, 0.5)
plt.xlabel('position in the window  [fraction of the output sample rate]')
plt.ylabel('amplitude [dB]')
plt.title('The corrected curve on its own - note the scale')
plt.grid(True, alpha=0.4); plt.tight_layout(); plt.show()


---
## Step 3 — Does the correction change the overall scale?

The coefficients sum to exactly 2¹⁷, which makes the filter's gain at zero frequency exactly 1.
That is deliberate: switching the correction on and off must change the *shape* of the response
and not its *size*, or every amplitude calibration would depend on a setting buried in a
register.

### Where this is measured, and why it matters

Amplitude here is worked out from the spread of a few thousand register reads, and how precise
that is depends strongly on how fast the signal is going round:

| beat frequency | 1σ on a ratio | why |
|---|---|---|
| 145 Hz | **0.06 %** | reads land at evenly spread phases; errors cancel |
| 500 Hz | 5.9 % | half the read rate — every read lands at nearly the same phase |
| 1831 Hz | 0.58 % | past the read rate, so phases are effectively random |
| 5000 Hz | 0.89 % | as above |

This step therefore measures **just off the centre, at 0.001 of the output rate** — about 120 Hz,
in the good column, and a frequency at which the designed filter's gain is 1.0000 to four
decimal places. It is a measurement of the gain at zero frequency in all but name, taken where
the measuring is ten times better.

### The tolerance is measured, not assumed

It used to be a flat 0.1 %, picked from the arithmetic — the coefficients sum to exactly 2¹⁷, so
the answer is exactly 1, so 0.1 % is surely generous. That reasoning is about the *filter* and
says nothing about the *measurement*. Taken at 1831 Hz, where one reading is worth 0.58 %, a
0.1 % threshold could only ever pass by luck; a run reporting 0.588 % was **1.02σ** — a
completely ordinary result failing an impossible test.

So the step now measures the same ratio several times and sets its own tolerance from the
scatter it actually sees. If the measurement is good the test is strict; if conditions are poor
it says so rather than pretending.

**Passes if:** the ratio is 1 to within three times its own measured uncertainty.


In [ ]:
# The designed filter's gain at this offset is 1.0000 to four decimal places, so
# any departure from 1 is either the coefficients or the measurement - and
# repeating it tells us which.
U_TEST = 0.001                      # ~120 Hz at R = 1024: the precise regime
REPEATS = 3

print('measuring the gain at u = +-%.3f  (%.0f Hz), %d times over'
      % (U_TEST, U_TEST * f_out, REPEATS))
print('')
print('  %8s %16s %16s %10s' % ('u', 'off', 'on', 'on/off'))
print('  ' + '-' * 54)

ratios = []
for rep in range(REPEATS):
    for u in (-U_TEST, +U_TEST):
        f_lo = F_TRUE - u * f_out
        aoff, _, _, _, _ = measure(f_lo, DEC, True)
        aon,  _, _, _, _ = measure(f_lo, DEC, False)
        r = aon / max(aoff, 1e-9)
        ratios.append(r)
        print('  %8.3f %16.1f %16.1f %10.6f' % (u, aoff, aon, r))

ratios = np.array(ratios)
ratio  = float(ratios.mean())
sigma  = float(ratios.std(ddof=1) / np.sqrt(len(ratios)))   # standard error
tol    = max(3 * sigma, 1e-3)

print('')
print('  mean ratio                 : %.6f   (want 1.000000)' % ratio)
print('  error                      : %+.4f %%' % (100 * (ratio - 1)))
print('  spread of the %d readings   : %.4f %% (1 sigma on one reading)'
      % (len(ratios), 100 * ratios.std(ddof=1)))
print('  uncertainty on the mean    : %.4f %%' % (100 * sigma))
print('  tolerance (3 sigma)        : %.4f %%' % (100 * tol))
print('')

ok = abs(ratio - 1) < tol
print('STEP 3:', 'PASS' if ok else 'FAIL')
if ok:
    print('   The correction changes the shape of the response without changing')
    print('   its size, to within what this measurement can resolve.')
else:
    print('   The ratio is outside its own measured uncertainty, so this is not')
    print('   scatter. Check the coefficient sum Step 1b printed - it should be')
    print('   exactly 131072. If it is, the fault is between the coefficients and')
    print('   the output: run Test Stage 5b, whose Test B measures this same gain')
    print('   with a single tap and a known answer.')


---
## Stage 5 summary

| Step | What it proves |
|---|---|
| 1 | The filter is genuinely in circuit, and bypasses itself when the decimation is too fast for it |
| 1b | The chip holds the intended coefficients — checked directly, not inferred from a signal |
| 2 | The window sags by ~2.3 dB uncorrected and is flat corrected — measured, not assumed |
| 3 | The correction changes the shape of the response without changing its scale |

Step 1b is the one to trust when the others disagree. It reads the hardware directly and
depends on nothing else working.

With this passing, the whole signal-processing chain is verified: reference tone, frequency
shift, averaging, and now correction. What remains is getting the data out of the chip, which
is Stages 6 and 7.

**Before moving on:**


In [ ]:
acq.disable()
print('acquisition disabled, scope restored to its normal inputs')